In [1]:
import pandas as pd
import numpy as np
import surprise
from surprise.model_selection import train_test_split

In [3]:
ratings = pd.read_csv("../filmtrust/ratings.txt",sep=" " ,names =['uid','iid','rating'])
ratings.head()

,uid,iid,rating
0,1,1,2.0
1,1,2,4.0
2,1,3,3.5
3,1,4,3.0
4,1,5,4.0


In [4]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings["rating"].max()
lowest_rating, highest_rating

(np.float64(0.5), np.float64(4.0))

In [5]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

similarity_options  = {"name":"cosine","user_based":True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())



Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
from surprise.model_selection import train_test_split




In [7]:
param_grid = {"n_epochs":[50],
              'n_factors':[100,50],
             'reg_all':np.linspace(0.01,0.8,5)}
kfold = KFold(n_splits=5, random_state=26, shuffle=True)
gs = GridSearchCV(surprise.SVD,param_grid,measures = ["rmse","mae"],cv=kfold, joblib_verbose=3,n_jobs=-1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  36 out of  50 | elapsed:    5.4s remaining:    2.0s
[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:    6.0s finished


In [8]:
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])


0.801940787741881
{'n_epochs': 50, 'n_factors': 100, 'reg_all': np.float64(0.20750000000000002)}


In [9]:
# Building bets model on whole data
algo =gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

In [10]:
iids = ratings["iid"].unique()
iids

array([   1,    2,    3, ..., 2069, 2070, 2071], shape=(2071,))

In [11]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [12]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print("List of items rated by user : \n",u_iid)
print("\nNo of items rated by user {0} : {1} ".format(user, len(u_iid)))
iids_ro_pred = np.setdiff1d(iids, u_iid)
print("\nItems not rated by the user or those items for which the expected ratings are to be predicted : ", iids_ro_pred)

List of items rated by user : 
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]

No of items rated by user 50 : 39 

Items not rated by the user or those items for which the expected ratings are to be predicted :  [  14   15   16 ... 2069 2070 2071]


In [13]:
testset = [[user,iid,0.] for iid in iids_ro_pred]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
109,126,4.000000
485,525,4.000000
172,189,4.000000
1078,1118,4.000000
649,689,4.000000
53,68,4.000000
246,286,4.000000
374,414,4.000000
765,805,4.000000
730,770,4.000000
